# Day 4: SFT 监督微调（Supervised Fine-Tuning）

学习目标：
1. 理解 Base Model 与 Chat Model 的区别
2. 掌握 SFT 数据格式和 Chat Template
3. 理解 Loss Masking 的原理和重要性
4. 理解 LoRA 参数高效微调原理
5. 用 SFTTrainer + LoRA 完成一次中文指令微调

## 1. Base Model vs Chat Model

### 核心区别

| 特性 | Base Model | Chat Model |
|------|-----------|------------|
| 训练方式 | 纯文本预训练 | 预训练 + SFT + RLHF |
| 行为模式 | 续写文本 | 遵循指令、对话 |
| 输入格式 | 任意文本 | Chat Template 格式 |
| 典型表现 | 可能重复、跑题 | 结构化回答 |

Base Model 本质上是一个**文本续写器**，不会"回答问题"。

### 1.1 演示 Base Model 的行为

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2-0.5B"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {total_params:,}")

In [ ]:
# Base model 的 "奇怪" 行为：它不会回答问题，只会续写
def generate(model, prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, do_sample=True, top_k=50,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# 直接问问题 -> Base model 会续写而非回答
test_prompts = [
    "请解释什么是机器学习？",
    "1+1等于几？",
    "写一首关于春天的诗。",
]

print("Base Model 的行为（直接输入问题）:")
print("=" * 60)
for prompt in test_prompts:
    response = generate(model, prompt, max_new_tokens=80)
    print(f"输入: {prompt}")
    print(f"输出: {response[:200]}")
    print("-" * 60)

可以看到 Base Model 倾向于"续写"输入文本，而非像 ChatGPT 那样"回答"问题。

这就是为什么需要 **SFT（监督微调）**：教模型学会"回答问题"的格式。

## 2. SFT 数据格式

### 2.1 Alpaca 格式

SFT 数据的标准格式包含 instruction（指令）、input（可选输入）和 output（期望输出）。

In [ ]:
# 展示一条真实的 SFT 数据样本
sample = {
    "instruction": "解释什么是梯度下降算法？",
    "input": "",
    "output": "梯度下降是一种优化算法，核心思想是沿着损失函数梯度的反方向更新参数..."
}

import json
print("Alpaca 格式样本:")
print(json.dumps(sample, ensure_ascii=False, indent=2))

### 2.2 Chat Template 格式

现代 LLM 使用 Chat Template 来区分对话中的不同角色。

In [ ]:
# Qwen2 的 Chat Template
messages = [
    {"role": "user", "content": "什么是机器学习？"},
    {"role": "assistant", "content": "机器学习是人工智能的一个分支..."},
]

formatted = tokenizer.apply_chat_template(messages, tokenize=False)
print("Chat Template 格式化结果:")
print(formatted)
print()
print("注意：Qwen2 使用 <|im_start|> 和 <|im_end|> 来标记角色边界")

In [ ]:
# 将 Alpaca 格式转换为 Chat 格式
def alpaca_to_chat(instruction, input_text, output):
    user_content = f"{instruction}\n\n{input_text}" if input_text else instruction
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

# 示例
result = alpaca_to_chat("解释什么是 Python", "", "Python 是一种高级编程语言...")
print("转换后:")
print(result)

## 3. Loss Masking 详解

### 为什么只在 assistant 部分算 loss？

在 SFT 中，我们的目标是让模型学会"怎么回答"，而不是"怎么提问"。

```
<|im_start|>user           ← 这部分的 label 设为 -100（忽略）
什么是机器学习？            ← 这部分的 label 设为 -100（忽略）
<|im_end|>                  ← 这部分的 label 设为 -100（忽略）
<|im_start|>assistant       ← 这部分的 label 设为 -100（忽略）
机器学习是...               ← 只在这部分计算 loss！
<|im_end|>                  ← 计算 loss
```

**原理**：将不需要学习的 token 的 label 设为 -100（PyTorch CrossEntropyLoss 的 ignore_index）

In [ ]:
# 演示 Loss Masking 的效果
import torch.nn.functional as F

# 模拟一个简单的例子
messages = [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好！有什么可以帮你的？"},
]

formatted = tokenizer.apply_chat_template(messages, tokenize=False)
tokens = tokenizer.encode(formatted)
token_strs = tokenizer.convert_ids_to_tokens(tokens)

print("Token 序列和 Loss Mask:")
print(f"完整文本: {formatted}")
print(f"\nToken 数量: {len(tokens)}")
print()

# 找到 assistant 部分的起始位置
assistant_start = formatted.find("有什么")
assistant_tokens_start = len(tokenizer.encode(formatted[:assistant_start]))

for i, (tid, tstr) in enumerate(zip(tokens, token_strs)):
    mask = "COMPUTE" if i >= assistant_tokens_start else "IGNORE "
    print(f"  [{mask}] {i:3d}: {tid:6d} -> {tstr}")

## 4. LoRA 原理

### 核心思想：W = W₀ + A × B

- **W₀**: 原始预训练权重（冻结不更新）
- **A** (d×r) 和 **B** (r×d): 低秩分解矩阵（可训练）
- **r**: 秩（rank），远小于 d

```
原始: W (4096 × 4096) = 16M 参数
LoRA: A (4096 × 16) × B (16 × 4096) = 131K 参数
参数减少: 99.2%
```

### 为什么 LoRA 有效？
1. 微调时权重的变化通常是低秩的
2. 大部分预训练知识保留在 W₀ 中
3. 推理时可以合并 W₀ + AB，无额外延迟

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# LoRA 配置
lora_config = LoraConfig(
    r=16,                           # 低秩矩阵的秩
    lora_alpha=32,                  # 缩放因子 (alpha/r 决定影响程度)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# 查看参数量变化
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n可训练参数: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# 查看哪些层被添加了 LoRA
print("LoRA 层:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

## 5. SFTTrainer 实战

### 5.1 加载训练数据

In [ ]:
from datasets import load_dataset

# 加载中文 Alpaca 数据集
dataset = load_dataset("silk-road/alpaca-data-gpt4-chinese", trust_remote_code=True)
print(f"数据集大小: {len(dataset['train'])}")
print(f"\n样本示例:")
print(json.dumps(dataset['train'][0], ensure_ascii=False, indent=2))

# 取子集
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
print(f"\n使用训练数据: {len(train_dataset)} 条")

In [ ]:
# 格式化数据
def format_alpaca_to_chat(example):
    instruction = example["instruction"]
    input_text = example.get("input", "")
    output = example["output"]
    user_content = f"{instruction}\n\n{input_text}" if input_text else instruction
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_dataset = train_dataset.map(format_alpaca_to_chat, remove_columns=train_dataset.column_names)
print(f"格式化完成! 样本:")
print(train_dataset[0]['text'][:300])

### 5.2 训练前生成

In [ ]:
# 训练前先看 base model 的回复
test_prompts = [
    "请解释什么是机器学习？",
    "写一首关于春天的诗。",
    "如何学好编程？",
    "Python 和 Java 哪个更适合初学者？",
    "什么是深度学习？",
]

print("训练前 Base Model 的回复:")
print("=" * 60)
pre_sft_responses = []
model.eval()
for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=150, temperature=0.7,
            do_sample=True, top_k=50, pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    pre_sft_responses.append(response)
    print(f"\n问: {prompt}")
    print(f"答: {response[:200]}")

### 5.3 配置 SFTTrainer 并训练

In [ ]:
from trl import SFTTrainer, SFTConfig
import time

sft_config = SFTConfig(
    output_dir="./sft_output",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,       # 等效 batch_size = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=20,
    save_strategy="epoch",
    max_seq_length=512,
    fp16=torch.cuda.is_available(),
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("开始 SFT 训练...")
start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time
print(f"\n训练完成! 耗时: {elapsed:.0f}s")
print(f"训练 Loss: {train_result.metrics['train_loss']:.4f}")

### 5.4 训练后生成

In [ ]:
# 训练后的回复
print("训练后 SFT Model 的回复:")
print("=" * 60)
post_sft_responses = []
model.eval()
for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=150, temperature=0.7,
            do_sample=True, top_k=50, pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    post_sft_responses.append(response)
    print(f"\n问: {prompt}")
    print(f"答: {response[:200]}")

## 6. Base vs SFT 对比

In [ ]:
# 并排对比
print("Base Model vs SFT Model 对比")
print("=" * 80)
for i, prompt in enumerate(test_prompts):
    print(f"\n问: {prompt}")
    print(f"\n  [Base Model]:")
    print(f"  {pre_sft_responses[i][:200]}")
    print(f"\n  [SFT Model]:")
    print(f"  {post_sft_responses[i][:200]}")
    print("-" * 80)

## 7. 总结

### 今日要点

1. **Base vs Chat**: Base Model 是文本续写器，SFT 让它学会"对话"
2. **数据格式**: Alpaca 格式 -> Chat Template 格式的转换
3. **Loss Masking**: 只在 assistant 回复部分计算 loss，提高学习效率
4. **LoRA**: W = W₀ + AB，只训练 <1% 的参数就能有效微调
5. **SFTTrainer**: trl 库的高层 API，集成了数据处理和训练逻辑

### 思考题

1. 为什么 LoRA 的 rank (r) 不能设得太大或太小？
2. Loss Masking 如果不做会怎样？对训练效果有什么影响？
3. SFT 数据中，instruction 和 output 的长度比例对训练有什么影响？
4. 为什么 SFT 通常只训练 1-3 个 epoch？
5. LoRA 的 target_modules 应该如何选择？